In [1]:
import sys
print(sys.executable)

C:\Users\user\anaconda3\envs\ragtraining\python.exe


In [2]:
import chromadb

client = chromadb.PersistentClient(path="./vectorstores/alice")
collection = client.get_collection("alice")

# 查看資料庫內所有資料
results = collection.get(include=["documents", "metadatas"])

print(f"共有 {len(results['ids'])} 筆資料")
for i, (id_, doc, meta) in enumerate(zip(results['ids'], results['documents'], results['metadatas'])):
    print(f"\n[{i+1}] id: {id_}")
    print(f"     type: {meta.get('type')}")
    print(f"     source: {meta.get('source')}")
    print(f"     document: {doc[:100]}")

共有 20 筆資料

[1] id: a.txt
     type: text
     source: docs\a.txt
     document: 粒線體是細胞的發電廠

[2] id: b.txt
     type: text
     source: docs\b.txt
     document: 1+1會等於2

[3] id: Apple.jpg#0
     type: image
     source: images\Apple.jpg
     document: 這是一份生物科的筆記。

這張圖片展示了一個單獨的紅色蘋果，其外觀呈現出左右反轉的效果。蘋果的整體形狀是經典的圓錐形，底部較寬，向上逐漸收窄。它的表皮呈現出鮮豔的深紅色，帶有一些自然的光澤，看起來光滑

[4] id: Apple.jpg#1
     type: image
     source: images\Apple.jpg
     document: ，連接著一根短小的深棕色果梗。從這根果梗上，向左上方伸展出一片翠綠色的葉子。這片葉子形狀完整，邊緣可能帶有輕微的鋸齒，葉脈清晰可見。蘋果的表面可能有一些細微的顏色變化，例如在某些區域呈現出較淺的紅色或

[5] id: Banana.jpg#0
     type: image
     source: images\Banana.jpg
     document: 這是一份【生物】科的筆記。
一張單一的、成熟的黃色香蕉放置在木質表面上。香蕉的莖部位於圖片的右側，其身體向左側輕柔彎曲，尖端指向左邊。香蕉呈現光滑的質地和鮮豔的黃色，左側尖端有一個小小的深色斑點。它在

[6] id: Banana.jpg#1
     type: image
     source: images\Banana.jpg
     document: 背景是一個溫暖的棕色木質表面，由幾塊木板組成。木紋清晰可見，橫向貫穿整個圖片，木材的色調和紋理有著細微的變化。

[7] id: Civics_Note.jpg#0
     type: image
     source: images\Civics_Note.jpg
     document: 這是一份公民科的筆記。

在圖片的左側，首先可以看到標題「比例原則」。其下方列出了

In [ ]:
import os
from dotenv import load_dotenv

# 從 .env 載入機密金鑰，避免硬編碼在程式碼中
load_dotenv()

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"

# 機密金鑰由 .env 提供
for _key in ("LANGCHAIN_API_KEY", "GEMINI_API_KEY_PRIMARY", "GEMINI_API_KEY_SECONDARY"):
    if not os.environ.get(_key):
        raise RuntimeError(f"缺少環境變數 {_key}，請在 .env 檔中設定")

# Gemini 有兩把金鑰以分散用量限制：
#   主要金鑰 (預設使用) / 備用金鑰 (主要金鑰用完額度時可切換)
gemini_api_key_primary = os.environ["GEMINI_API_KEY_PRIMARY"]
gemini_api_key_secondary = os.environ["GEMINI_API_KEY_SECONDARY"]

# 預設使用主要金鑰
active_gemini_api_key = gemini_api_key_primary

import torch
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI

# 有 GPU 就用 GPU，否則退回 CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用裝置: {device}")

# 載入檢索用嵌入模型 intfloat/multilingual-e5-base（繁中+英文皆強，跨語言檢索佳），
# 取代原本對中文較弱且只看前 77 個 token 的 CLIP 文字編碼器。
# E5 需要前綴：文件用 "passage: "，查詢用 "query: "。
# 只載入一次，避免重複執行 cell 時重新建立模型。
EMBED_MODEL = "intfloat/multilingual-e5-base"
if "embedder" not in globals():
    embedder = SentenceTransformer(EMBED_MODEL, device=device)

# 用 E5 embed 查詢文字（query 端）
def embed_query(text: str):
    return embedder.encode([f"query: {text}"], normalize_embeddings=True)[0].tolist()

# 連接對應使用者的資料庫
user_name = "alice"
client = chromadb.PersistentClient(path=f"./vectorstores/{user_name}")
vector_collection = client.get_collection(user_name)  # collection 名稱 = user_name

# Gemini 生成模型（預設使用主要金鑰，可改成 gemini_api_key_secondary 切換備用金鑰）
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    google_api_key=active_gemini_api_key
)

def content_to_text(content) -> str:
    """把 LLM 回傳的 content 轉成純文字。
    gemini-3.1-flash-lite 可能回傳結構化的 content blocks（list of dict/str），
    而非單純字串，因此需要把各區塊的文字串接起來。"""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type", "text") == "text":
                parts.append(block.get("text", ""))
        return "".join(parts)
    return str(content)

def rag_query(user_question, top_k=5):
    # Step 1: 用 E5 embed 問題（query 前綴）
    q_embedding = embed_query(user_question)

    # Step 2: 搜尋相關文件
    results = vector_collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k
    )

    # Step 3: 取出文件內容
    docs = results["documents"][0]
    context = "\n---\n".join(docs)

    # Step 4: 組成 Prompt 呼叫 Gemini
    prompt = f"""你是一個問答助手，請根據以下提供的資料盡量完整且詳細的回答問題。
如果資料中沒有相關資訊，請回答「參考資料中沒有記載相關訊息」，不要自行編造。
注意: 1.請勿使用粗體 2.請勿套用latex格式 3.你得到的資料是基於原始資料的敘述，若敘述中含有使用者提問的項目，請將其統整後再回答。

【參考資料】
{context}

【使用者問題】
{user_question}

【回答】"""

    response = llm.invoke(prompt)
    return content_to_text(response.content)

input_question = input('輸入問題: ')
answer = rag_query(
    user_question=input_question,
    top_k=10
)
print(answer)